In [ ]:
import pandas as pd
import dai
from datetime import date, timedelta
import bigtrader
# 导入包
from bigmodule import M

# <aistudiograph>

# @param(id="m4", name="initialize")
# 交易引擎：初始化函数，只执行一次
def m4_initialize_bigquant_run(context):    
    import math

    try:
        from bigtrader.finance import PerOrder
    except (ImportError, ModuleNotFoundError):
        from bigtrader.finance.commission import PerOrder
    import numpy as np
    # 系统已经设置了默认的交易手续费和滑点，要修改手续费可使用如下函数
    context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))
    # 设置买入的股票数量，这里买入预测股票列表排名靠前的5只
    stock_count = 5
    # 每只的股票的权重，如下的权重分配会使得靠前的股票分配多一点的资金，[0.339160, 0.213986, 0.169580, ..]
    context.stock_weights = np.array([1/np.log(i+2) for i in range(stock_count)])
    context.stock_weights = context.stock_weights / np.sum(context.stock_weights)
    # 设置每只股票占用的最大资金比例
    context.max_cash_per_instrument = 0.2
    context.options['hold_days'] = 5


# @param(id="m4", name="before_trading_start")
# 交易引擎：每个单位时间开盘前调用一次。
def m4_before_trading_start_bigquant_run(context, data):
    # 盘前处理，订阅行情等  
    # 初始化股票池 
    yesterday = date.today() - timedelta(days=1)
    day_str = yesterday.strftime('%Y-%m-%d')
    stock_pool = dai.query("SELECT * FROM stock_pool_wyc WHERE date = '{}'".format(day_str)).df()
    instruments = stock_pool['code'].astype(str).unique().tolist()

    if not instruments:
        context.stock_pool = []
        return

    try:
        context.subscribe(instruments)
    except Exception:
        pass

    hist = data.history(instruments, ['close'], 90, '1d')
    if isinstance(hist, pd.DataFrame) and 'close' in hist.columns:
        close_df = hist['close']
    else:
        close_df = hist

    qualified = []
    for instrument in instruments:
        if instrument not in close_df.columns:
            continue
        close_series = close_df[instrument].dropna()
        if len(close_series) < 90:
            continue

        ma60 = close_series.rolling(60).mean()
        recent_close = close_series.iloc[-30:]
        recent_ma60 = ma60.iloc[-30:]
        if len(recent_close) < 30 or recent_ma60.isna().any():
            continue

        if (recent_close < recent_ma60).all():
            qualified.append(instrument)

    context.stock_pool = qualified
    pass


# @param(id="m4", name="handle_tick")
# 交易引擎：tick数据处理函数，每个tick执行一次
def m4_handle_tick_bigquant_run(context, tick):
    pass

# @param(id="m4", name="handle_data")
def m4_handle_data_bigquant_run(context, data):
    # 按日期过滤得到今日的预测数据
    ranker_prediction = context.data[context.data.date == data.current_dt.strftime('%Y-%m-%d')]
    # 按照position排序
    ranker_prediction.sort_values(["date", "position"], inplace=True)
    ranker_prediction.reset_index(drop=True, inplace=True)

    # 1. 资金分配
    # 平均持仓时间是hold_days，每日都将买入股票，每日预期使用 1/hold_days 的资金
    # 实际操作中，会存在一定的买入误差，所以在前hold_days天，等量使用资金；之后，尽量使用剩余资金（这里设置最多用等量的1.5倍）
    is_staging = context.trading_day_index < context.options['hold_days'] # 是否在建仓期间（前 hold_days 天）
    cash_avg = context.portfolio.portfolio_value / context.options['hold_days']
    cash_for_buy = min(context.portfolio.cash, (1 if is_staging else 1.5) * cash_avg)
    cash_for_sell = cash_avg - (context.portfolio.cash - cash_for_buy)
    positions = {e: p.amount * p.last_sale_price
                for e, p in context.portfolio.positions.items()}

    # 2. 生成卖出订单：hold_days天之后才开始卖出；对持仓的股票，按机器学习算法预测的排序末位淘汰
    if not is_staging and cash_for_sell > 0:
        equities = {e: e for e, p in context.portfolio.positions.items()}
        instruments = list(reversed(list(ranker_prediction.instrument[ranker_prediction.instrument.apply(
                lambda x: x in equities)])))
        for instrument in instruments:
            context.order_target(instrument, 0)
            cash_for_sell -= positions[instrument]
            if cash_for_sell <= 0:
                break

    # 3. 生成买入订单：按机器学习算法预测的排序，买入前面的stock_count只股票
    buy_cash_weights = context.stock_weights
    buy_instruments = list(ranker_prediction.instrument[:len(buy_cash_weights)])
    max_cash_per_instrument = context.portfolio.portfolio_value * context.max_cash_per_instrument
    for i, instrument in enumerate(buy_instruments):
        cash = cash_for_buy * buy_cash_weights[i]
        if cash > max_cash_per_instrument - positions.get(instrument, 0):
            # 确保股票持仓量不会超过每次股票最大的占用资金量
            cash = max_cash_per_instrument - positions.get(instrument, 0)
        if cash > 0:
            context.order_value(instrument, cash)

# @param(id="m4", name="handle_trade")
# 交易引擎：成交回报处理函数，每个成交发生时执行一次
def m4_handle_trade_bigquant_run(context, trade):
    pass

# @param(id="m4", name="handle_order")
# 交易引擎：委托回报处理函数，每个委托变化时执行一次
def m4_handle_order_bigquant_run(context, order):
    pass

# @param(id="m4", name="after_trading")
# 交易引擎：盘后处理函数，每日盘后执行一次
def m4_after_trading_bigquant_run(context, data):
    pass

# @module(position="-392,-228", comment="""通过SQL调用股票池数据""")
m1 = M.input_features_dai.v30(
    mode="""表达式""",
    expr="""
        -- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
        -- 数据&字段: 数据文档 https://bigquant.com/data/home
        -- 数据使用: 表名.字段名, 对于没有指定表名的列，会从 expr_tables 推断
        -- 给输出数据列命名: AS field_name
        -- 在这里输入表达式, 每行一个表达式, 会根据这个输入解析表名并构建查询和计算SQL,

        stock_pool_wyc.date
        stock_pool_wyc.code
        stock_pool_wyc.name
    """,
    expr_filters="""
        -- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
        -- 数据&字段: 数据文档 https://bigquant.com/data/home
        -- 表达式模式的过滤都是放在 QUALIFY 里, 即数据查询、计算, 最后才到过滤条件

        -- c_pct_rank(-return_90) <= 0.3
        -- c_pct_rank(return_30) <= 0.3
        -- cn_stock_bar1d.turn > 0.02
    """,
    expr_tables="""stock_pool_wyc""",
    extra_fields="""date, code""",
    order_by="""date""",
    expr_drop_na=True,
    sql="""SELECT * FROM stock_pool_wyc ORDER BY date, code""",
    extract_data=True,
    m_name="""m1"""
)

# @module(position="-341,-3", comment="""抽取数据，设置数据开始时间和结束时间，并绑定模拟交易""")
m2 = M.extract_data_dai.v20(
    sql=m1.data,
    start_date="""2021-12-06""",
    start_date_bound_to_trading_date=True,
    end_date="""2024-04-10""",
    end_date_bound_to_trading_date=True,
    before_start_days=90,
    keep_before=False,
    debug=False,
    m_name="""m2"""
)

# @module(position="-254,139", comment="""交易，日线，设置初始化函数和K线处理函数，以及初始资金、基准等""")
m4 = M.bigtrader.v58(
    data=m2.data,
    start_date="""""",
    end_date="""""",
    initialize=m4_initialize_bigquant_run,
    before_trading_start=m4_before_trading_start_bigquant_run,
    handle_tick=m4_handle_tick_bigquant_run,
    handle_data=m4_handle_data_bigquant_run,
    handle_trade=m4_handle_trade_bigquant_run,
    handle_order=m4_handle_order_bigquant_run,
    after_trading=m4_after_trading_bigquant_run,
    capital_base=1000000,
    frequency="""daily""",
    product_type="""股票""",
    rebalance_period_type="""交易日""",
    rebalance_period_days="""1""",
    rebalance_period_roll_forward=True,
    backtest_engine_mode="""标准模式""",
    before_start_days=0,
    volume_limit=1,
    order_price_field_buy="""open""",
    order_price_field_sell="""close""",
    benchmark="""000300.SH""",
    plot_charts="""True""",
    debug=False,
    backtest_only=False,
    m_cached=False,
    m_name="""m4"""
)
# </aistudiograph>

[2026-04-23 17:37:46] [info     ] input_features_dai.v30 开始运行 ..
[2026-04-23 17:37:46] [info     ] expr mode
[2026-04-23 17:37:46] [info     ] extract data ..
[2026-04-23 17:37:46] [info     ] extracted (816, 5).
[2026-04-23 17:37:46] [info     ] input_features_dai.v30 运行完成 [0.233s].
